In [1]:
import pandas as pd
import numpy as np
import pickle

prices = pd.read_csv("../data/processed/prices.csv", index_col="date", parse_dates=True)
with open("../data/processed/pair_signals.pkl", "rb") as f:
    all_signals = pickle.load(f)

selected_pairs = pd.read_csv("../data/processed/selected_pairs.csv")
print(f"Loaded signals for {len(all_signals)} pairs")

Loaded signals for 87 pairs


In [2]:
def generate_positions(z_score, entry=1.0, exit=0.2):
    """
    Returns a position series: +1 = long coin_a / short coin_b,
    -1 = short coin_a / long coin_b, 0 = flat.
    """
    position = pd.Series(0, index=z_score.index)
    pos = 0
    for i in range(len(z_score)):
        z = z_score.iloc[i]
        if pd.isna(z):
            position.iloc[i] = 0
            continue
        if pos == 0:
            if z > entry:
                pos = -1   # spread too high -> bet it falls back: short a, long b
            elif z < -entry:
                pos = 1    # spread too low -> bet it rises back: long a, short b
        else:
            if abs(z) < exit:
                pos = 0    # spread back near normal -> close out
        position.iloc[i] = pos
    return position

In [3]:
EXIT_THRESHOLDS = [0.1, 0.2, 0.5, 0.7]

def summarize_positions(position):
    """Quick stats: number of trades entered, average holding length."""
    changes = position.diff().fillna(0) != 0
    n_trades = (changes & (position != 0)).sum()
    days_in_position = (position != 0).sum()
    avg_holding = days_in_position / n_trades if n_trades > 0 else 0
    return n_trades, avg_holding

# test on the single best pair by cointegration p-value first
best_pair = tuple(selected_pairs.sort_values("p_value").iloc[0][["coin_a", "coin_b"]])
z = all_signals[best_pair]["z_score"]

print(f"Pair: {best_pair}")
for exit_thresh in EXIT_THRESHOLDS:
    pos = generate_positions(z, entry=1.0, exit=exit_thresh)
    n_trades, avg_holding = summarize_positions(pos)
    print(f"  exit={exit_thresh}: {n_trades} trades, {avg_holding:.1f} days avg holding")

Pair: ('havven', 'jasmycoin')
  exit=0.1: 12 trades, 31.7 days avg holding
  exit=0.2: 16 trades, 21.6 days avg holding
  exit=0.5: 22 trades, 11.3 days avg holding
  exit=0.7: 22 trades, 10.3 days avg holding


In [4]:
EXIT_THRESHOLD = 0.2  # a reasonable middle ground; revisit after Phase 8's comparison

all_positions = {}
for pair, sig in all_signals.items():
    all_positions[pair] = generate_positions(sig["z_score"], entry=1.0, exit=EXIT_THRESHOLD)

print(f"Generated positions for {len(all_positions)} pairs")

Generated positions for 87 pairs


In [9]:
def compute_pair_returns(price_a, price_b, position, beta):
    """Daily P&L for a dollar-neutral pair position, sized by hedge ratio."""
    ret_a = price_a.pct_change(fill_method=None)
    ret_b = price_b.pct_change(fill_method=None)
    # long/short a, hedge-ratio-weighted short/long b
    pair_return = position.shift(1) * (ret_a - beta.shift(1) * ret_b)
    return pair_return

pair_returns = {}
for pair, sig in all_signals.items():
    a, b = pair
    pair_returns[pair] = compute_pair_returns(prices[a], prices[b], all_positions[pair], sig["beta"])

# equal-weight across all 87 pairs for now
portfolio_returns = pd.DataFrame(pair_returns).mean(axis=1)

In [10]:
portfolio_returns.describe()

count    641.000000
mean       0.000390
std        0.006692
min       -0.081378
25%       -0.001801
50%        0.000000
75%        0.003458
max        0.048569
dtype: float64

In [11]:
portfolio_returns.dropna().head()

date
2024-11-25    0.0
2024-11-26    0.0
2024-11-27    0.0
2024-11-28    0.0
2024-11-29    0.0
dtype: float64

In [12]:
portfolio_returns.dropna()[portfolio_returns.dropna() != 0].head()

date
2025-02-22    0.005284
2025-02-23   -0.006926
2025-02-24   -0.020068
2025-02-25    0.006138
2025-02-26   -0.005085
dtype: float64

In [13]:
portfolio_returns.to_csv("../data/processed/portfolio_returns.csv")

In [14]:
COST_BPS = 0.0020  # 20 bps all-in, per unit of position change

def compute_pair_costs(position, cost_bps=COST_BPS):
    """Cost incurred each day a position changes, sized by how much it changed."""
    position_change = position.diff().abs().fillna(0)
    return position_change * cost_bps

pair_costs = {}
for pair, pos in all_positions.items():
    pair_costs[pair] = compute_pair_costs(pos)

portfolio_costs = pd.DataFrame(pair_costs).mean(axis=1)
portfolio_costs.describe()

count    731.000000
mean       0.000081
std        0.000084
min        0.000000
25%        0.000000
50%        0.000069
75%        0.000115
max        0.000874
dtype: float64

In [15]:
portfolio_returns_net = portfolio_returns - portfolio_costs
portfolio_returns_net.describe()

count    641.000000
mean       0.000297
std        0.006687
min       -0.081538
25%       -0.001940
50%        0.000000
75%        0.003320
max        0.048523
dtype: float64

In [16]:
def pair_turnover_stats(position):
    changes = (position.diff().abs().fillna(0) > 0)
    n_trades = changes.sum()
    days_active = (position != 0).sum()
    avg_holding = days_active / n_trades if n_trades > 0 else np.nan
    return n_trades, avg_holding

turnover_stats = []
for pair, pos in all_positions.items():
    n_trades, avg_holding = pair_turnover_stats(pos)
    turnover_stats.append({"pair": pair, "n_trades": n_trades, "avg_holding_days": avg_holding})

turnover_df = pd.DataFrame(turnover_stats)
turnover_df[["n_trades", "avg_holding_days"]].describe()

,n_trades,avg_holding_days
count,87.000000,87.000000
mean,29.781609,11.027120
std,8.495451,2.864308
min,4.000000,5.866667
25%,23.000000,8.845322
50%,30.000000,10.800000
75%,37.000000,12.516685
max,49.000000,19.052632


In [17]:
gross_annualized = portfolio_returns.mean() * 252
net_annualized = portfolio_returns_net.mean() * 252
total_cost_drag = portfolio_costs.mean() * 252

print(f"Gross annualized return (rough): {gross_annualized:.2%}")
print(f"Cost drag (rough, annualized):   -{total_cost_drag:.2%}")
print(f"Net annualized return (rough):   {net_annualized:.2%}")

Gross annualized return (rough): 9.83%
Cost drag (rough, annualized):   -2.05%
Net annualized return (rough):   7.49%


In [18]:
results = []
for exit_thresh in [0.1, 0.2, 0.5, 0.7]:
    positions_test = {pair: generate_positions(sig["z_score"], entry=1.0, exit=exit_thresh)
                       for pair, sig in all_signals.items()}
    returns_test = {pair: compute_pair_returns(prices[a], prices[b], positions_test[pair], all_signals[pair]["beta"])
                     for pair, sig in all_signals.items() for a, b in [pair]}
    costs_test = {pair: compute_pair_costs(pos) for pair, pos in positions_test.items()}

    port_ret = pd.DataFrame(returns_test).mean(axis=1)
    port_cost = pd.DataFrame(costs_test).mean(axis=1)
    net = port_ret - port_cost

    results.append({
        "exit": exit_thresh,
        "gross_annualized": port_ret.mean() * 252,
        "cost_drag": port_cost.mean() * 252,
        "net_annualized": net.mean() * 252,
    })

pd.DataFrame(results)

,exit,gross_annualized,cost_drag,net_annualized
0,0.1,0.063078,0.016088,0.044732
1,0.2,0.098301,0.020533,0.074885
2,0.5,0.150117,0.028466,0.117654
3,0.7,0.327531,0.034299,0.288416


In [19]:
results_full = []
for exit_thresh in [0.1, 0.2, 0.5, 0.7]:
    positions_test = {pair: generate_positions(sig["z_score"], entry=1.0, exit=exit_thresh)
                       for pair, sig in all_signals.items()}
    returns_test = {pair: compute_pair_returns(prices[a], prices[b], positions_test[pair], all_signals[pair]["beta"])
                     for pair, sig in all_signals.items() for a, b in [pair]}
    costs_test = {pair: compute_pair_costs(pos) for pair, pos in positions_test.items()}

    port_ret = pd.DataFrame(returns_test).mean(axis=1)
    port_cost = pd.DataFrame(costs_test).mean(axis=1)
    net = port_ret - port_cost

    ann_return = net.mean() * 252
    ann_vol = net.std() * np.sqrt(252)
    sharpe = ann_return / ann_vol if ann_vol > 0 else np.nan

    results_full.append({
        "exit": exit_thresh,
        "net_annualized": ann_return,
        "annualized_vol": ann_vol,
        "sharpe": sharpe,
        "max_daily_gain": net.max(),
        "max_daily_loss": net.min(),
    })

pd.DataFrame(results_full)

,exit,net_annualized,annualized_vol,sharpe,max_daily_gain,max_daily_loss
0,0.1,0.044732,0.100899,0.443334,0.045026,-0.084665
1,0.2,0.074885,0.106158,0.705410,0.048523,-0.081538
2,0.5,0.117654,0.111298,1.057112,0.048022,-0.082062
3,0.7,0.288416,0.215673,1.337283,0.310440,-0.034136


In [20]:
positions_07 = {pair: generate_positions(sig["z_score"], entry=1.0, exit=0.7) for pair, sig in all_signals.items()}
returns_07 = {pair: compute_pair_returns(prices[a], prices[b], positions_07[pair], all_signals[pair]["beta"])
              for pair, sig in all_signals.items() for a, b in [pair]}
returns_07_df = pd.DataFrame(returns_07)

worst_day = returns_07_df.sum(axis=1).idxmax()  # or the date matching max_daily_gain
returns_07_df.loc[worst_day].sort_values(ascending=False).head(5)

meta-2-2       zetachain        8.375551
               swell-network    8.365256
               milk-alliance    8.349713
bitcoin-cash   openeden         0.641556
kucoin-shares  vechain          0.054927
Name: 2025-08-18 00:00:00, dtype: float64

In [21]:
portfolio_returns_net.to_csv("../data/processed/portfolio_returns_net.csv")